In [ ]:
import numpy as np
from pathlib import Path

embedding_dir = Path("../data/embeddings/codefuse-ai/F2LLM-v2-0.6B")

all_embeddings = []
embedding_indexer = []

for f in embedding_dir.iterdir():
    if f.stem.endswith("index"):
        continue
    embs = np.load(f)
    # only load newsgroups with approx. 500 messages
    if len(embs) < 400 or len(embs) > 600:
        continue
    all_embeddings.extend(embs)
    embedding_indexer += [f.stem] * len(embs)

len(embedding_indexer), len(all_embeddings)

(10066, 10066)

In [ ]:
all_embeddings = np.array(all_embeddings)
all_embeddings.shape

(10066, 1024)

## Dimensionality reduction for plotting with UMAP
The UMAP algorithm reduces the high-dimensional document embeddings (1024 dimensions) to 2 dimensions, so we can plot them as x,y coordinates 

In [ ]:
import umap

fit = umap.UMAP()
umap_2d_embeddings = fit.fit_transform(all_embeddings)
umap_2d_embeddings.shape

(10066, 2)

In [ ]:
from collections import Counter

# We will use newsgroups and data source info in plot below
newsgroups_indexer = [file_stem.split("_")[0] for file_stem in embedding_indexer]
sources_indexer = [file_stem.split("_")[1] for file_stem in embedding_indexer]

print(f"Number of messages per source {Counter(sources_indexer)}")
print(f"Number of messages per newsgroup {Counter(newsgroups_indexer)}")

Number of messages per source Counter({'nwa': 5098, 'ia': 4968})
Number of messages per newsgroup Counter({'no.lisp': 983, 'no.mail.drift': 577, 'no.news.drift': 572, 'no.skole.diverse': 568, 'no.alt.marked.seksualitet': 550, 'no.folklore.overtro': 532, 'no.it.diverse': 527, 'no.video': 511, 'no.fag.sjukepleie': 500, 'no.kultur.folklore.diverse': 500, 'no.annonser.it.unix': 500, 'no.org.efn.diskusjon': 494, 'no.alt.irctreff': 493, 'no.psykolog': 486, 'no.hobby.diverse': 484, 'no.slekt.programmer': 459, 'no.annonser.it.nettverk': 450, 'no.uninett.diverse': 444, 'no.ai': 436})


In [ ]:
import plotly.graph_objects as go

symbol_map = {"nwa": "circle", "ia": "triangle-up"}
unique_newsgroups = sorted(set(newsgroups_indexer))
colors_plotly = [
    f"hsl({int(i * 360 / len(unique_newsgroups))}, 70%, 50%)"
    for i in range(len(unique_newsgroups))
]
color_map = dict(zip(unique_newsgroups, colors_plotly))

fig = go.Figure()

for source, symbol in symbol_map.items():
    for ng in unique_newsgroups:
        mask = np.array(
            [
                s == source and n == ng
                for s, n in zip(sources_indexer, newsgroups_indexer)
            ]
        )
        if not mask.any():
            continue
        fig.add_trace(
            go.Scattergl(
                x=umap_2d_embeddings[mask, 0],
                y=umap_2d_embeddings[mask, 1],
                mode="markers",
                marker=dict(size=6, color=color_map[ng], symbol=symbol, opacity=0.7),
                name=f"{ng} ({source})",
                text=np.array(embedding_indexer)[mask],
                hovertemplate="%{text}<extra></extra>",
            )
        )

fig.update_layout(
    title="Norwegian Usenet message embeddings (color=newsgroup, shape=source)",
    xaxis_title="UMAP 1",
    yaxis_title="UMAP 2",
    width=1000,
    height=700,
    legend=dict(font=dict(size=9)),
)
fig.show()